# Assistant RAG — ShuriX Tech

Ce notebook interroge `doc/shuri.pdf` avec une recherche vectorielle et `gpt-4.1-mini`. Les réponses sont fondées sur les extraits récupérés et incluent leurs pages sources.

Avant d'exécuter : ajoutez `OPENAI_API_KEY=...` dans `.env`. Vous pouvez remplacer le modèle avec `OPENAI_MODEL`.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(dotenv_path=Path('.env'))

PDF_PATH = Path('doc/shuri.pdf')
MODEL_NAME = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY est absente. Créez .env avec OPENAI_API_KEY=...')
if not PDF_PATH.is_file():
    raise FileNotFoundError(f'PDF introuvable : {PDF_PATH.resolve()}')

print(f'Modèle de génération : {MODEL_NAME}')
print(f'Modèle d’embeddings : {EMBEDDING_MODEL}')

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
chunks = splitter.split_documents(pages)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu', 'local_files_only': True},
    encode_kwargs={'normalize_embeddings': True},
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f'shurix_{uuid4().hex}',
)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

print(f'{len(pages)} pages chargées ; {len(chunks)} extraits indexés.')

In [ ]:
def answer(question: str) -> str:
    """Répond à partir des passages RAG les plus pertinents."""
    if not question.strip():
        raise ValueError('La question ne peut pas être vide.')

    documents = retriever.invoke(question)
    context = '\n\n'.join(
        f'[Page {doc.metadata.get("page", 0) + 1}] {doc.page_content}'
        for doc in documents
    )
    pages = sorted({doc.metadata.get('page', 0) + 1 for doc in documents})
    messages = [
        SystemMessage(content=(
            'Tu es l’assistant documentaire de ShuriX Tech. Réponds en français '
            'uniquement à partir du contexte. Si l’information est absente, dis-le '
            'clairement. N’invente aucun fait et reste concis.'
        )),
        HumanMessage(content=f'CONTEXTE :\n{context}\n\nQUESTION : {question}'),
    ]
    response = llm.invoke(messages)
    return f'{response.content}\n\nSources : ' + ', '.join(f'p. {page}' for page in pages)


question = 'Quelle est la mission de ShuriX Tech ?'
print(answer(question))

## Essais

Modifiez `question` dans la dernière cellule, par exemple :

- `Quels sont les deux axes majeurs de ShuriX Tech ?`
- `Qu’est-ce que le Malaikadrone Defender ?`
- `Qui a fondé ShuriX Tech ?`